# Task 1.3 — Bronze Ingestion: Discussion Posts + Course Catalog
## Notebook: 03_bronze_discussion_catalog

**Layer:** Bronze (raw ingestion)

**What this notebook does:**
- Ingests `discussion_posts.json` (NDJSON) → `bronze.discussion_bronze`
  - Stores full `raw_json` field alongside top-level parsed fields for flexibility
  - Adds `_source_file`, `_load_ts`, `_last_modified_ts`
- Ingests `course_catalog.xlsx` / `course_catalog.csv` → `bronze.course_catalog_bronze`
  - Uses pandas.read_excel() or pandas.read_csv() → spark.createDataFrame()
  - Full OVERWRITE on every run (monthly admin refresh)
  - Adds `_source_file`, `_load_ts`, `_last_modified_ts`

**Business Rules enforced:**
- No duplicate course_ids in catalog
- is_active must be 0 or 1 only
- avg_rating must be between 0 and 5
- course_id and author_student_id must not be NULL in discussion posts

**Run order:** Run all cells top-to-bottom. Safe to re-run (idempotent).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"

# --- Volume source paths ---
DISCUSSION_RAW_PATH  = f"/Volumes/{CATALOG}/{BRONZE}/discussion_posts_raw/"
CATALOG_RAW_PATH     = f"/Volumes/{CATALOG}/{BRONZE}/course_catalog_raw/"

# --- Target Delta table names (fully qualified) ---
DISCUSSION_BRONZE_TABLE = f"{CATALOG}.{BRONZE}.discussion_bronze"
CATALOG_BRONZE_TABLE    = f"{CATALOG}.{BRONZE}.course_catalog_bronze"
DQ_AUDIT_TABLE          = f"{CATALOG}.audit.dq_audit"

# --- File names inside volumes ---
CATALOG_XLSX_FILE = "course_catalog.xlsx"    # primary
CATALOG_CSV_FILE  = "course_catalog.csv"     # fallback if XLSX not available

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema for this session ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(BRONZE)

print(f"✅ Catalog          : {CATALOG}")
print(f"✅ Schema           : {BRONZE}")
print(f"✅ Discussion path  : {DISCUSSION_RAW_PATH}")
print(f"✅ Catalog raw path : {CATALOG_RAW_PATH}")

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType,
    BooleanType, TimestampType, DoubleType
)
from delta.tables import DeltaTable
import pandas as pd
import os

print("✅ Imports successful")

## Part A — Discussion Posts Bronze (`discussion_bronze`)

**Source:** `discussion_posts.json` (NDJSON) from Volume `discussion_posts_raw/`
**Method:** Spark JSON read — one JSON object per line

Key design decision: we store the full original JSON as a `raw_json` string column
alongside the top-level extracted fields. This preserves full flexibility for
Silver Task 2.1 which does the deep parsing with from_json() and StructType schema.
The raw_json field acts as a safety net if the Silver schema evolves.

In [0]:
# ============================================================
# cell 5 — explicit schema for discussion_posts.json
# ============================================================

discussion_schema = StructType([
    StructField("post_id",           StringType(),    nullable=False),
    StructField("thread_id",         StringType(),    nullable=True),
    StructField("course_id",         StringType(),    nullable=True),
    StructField("author_student_id", StringType(),    nullable=True),
    StructField("parent_post_id",    StringType(),    nullable=True),  # null = top-level post
    StructField("post_ts",           TimestampType(), nullable=True),
    StructField("content_length_chars", IntegerType(), nullable=True),
    StructField("has_attachment",    BooleanType(),   nullable=True),
    StructField("like_count",        IntegerType(),   nullable=True),
    StructField("reply_count",       IntegerType(),   nullable=True),
    StructField("is_instructor_post", BooleanType(),  nullable=True),
])

print("✅ discussion schema defined with", len(discussion_schema.fields), "fields")

In [0]:
# ============================================================
# cell 6 — read discussion ndjson in two passes
# ============================================================

# pass 1 — read each line as a raw string (one json object per line = ndjson)
raw_text_df = (
    spark.read
    .text(DISCUSSION_RAW_PATH)
    .withColumnRenamed("value", "raw_json")
    # extract post_id from raw text to use as join key
    .withColumn("post_id_key", F.get_json_object(F.col("raw_json"), "$.post_id"))
)

# pass 2 — read with typed schema
# extract _source_file here BEFORE the join to avoid ambiguous _metadata reference
parsed_df = (
    spark.read
    .format("json")
    .option("multiLine", "false")          # ndjson: one record per line
    .option("mode", "PERMISSIVE")          # don't fail on malformed rows
    .schema(discussion_schema)
    .load(DISCUSSION_RAW_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# join on post_id to attach raw_json to each typed row
discussion_df = (
    parsed_df
    .join(
        raw_text_df.select("post_id_key", "raw_json"),
        parsed_df["post_id"] == raw_text_df["post_id_key"],
        how="left"
    )
    .drop("post_id_key")
    # ── add bronze metadata columns ─────────────────────────
    .withColumn("_load_ts",          F.current_timestamp())
    .withColumn("_last_modified_ts", F.current_timestamp())
    .withColumn("_schema_version",   F.lit(SCHEMA_VERSION))
)

row_count = discussion_df.count()
print(f"ℹ  rows read from source: {row_count:,}")

# write to delta — append mode
(
    discussion_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DISCUSSION_BRONZE_TABLE)
)

print(f"✅ written to: {DISCUSSION_BRONZE_TABLE}")
print(f"   rows written: {row_count:,}")

In [0]:
# ============================================================
# CELL 7 — Verify discussion_bronze
# ============================================================

disc_df = spark.table(DISCUSSION_BRONZE_TABLE)

print("── discussion_bronze ─────────────────────────────")
print(f"Total rows   : {disc_df.count():,}")
print(f"Columns      : {len(disc_df.columns)}")
print()

print("── instructor vs student posts ───────────────────")
disc_df.groupBy("is_instructor_post").count().show()

print("── top-level vs reply posts ──────────────────────")
disc_df.withColumn(
    "post_type",
    F.when(F.col("parent_post_id").isNull(), "top_level").otherwise("reply")
).groupBy("post_type").count().show()

print("── raw_json sample (first 120 chars) ────────────")
disc_df.select(
    "post_id",
    F.substring("raw_json", 1, 120).alias("raw_json_preview")
).show(3, truncate=False)

print("── metadata sample ───────────────────────────────")
disc_df.select(
    "post_id", "_load_ts", "_last_modified_ts", "_schema_version"
).show(3, truncate=50)

## Part B — Course Catalog Bronze (`course_catalog_bronze`)

**Source:** `course_catalog.xlsx` (or `course_catalog.csv`) from Volume `course_catalog_raw/`
**Method:** pandas.read_excel() → spark.createDataFrame() — then write as Delta

Why pandas for XLSX?
- Spark has no native XLSX reader
- pandas + openpyxl handles .xlsx correctly including data types
- For CSV fallback: pandas.read_csv() works the same way

Write mode: OVERWRITE — the catalog is a full monthly admin refresh.
Every run replaces the entire table with the latest version from the file.

In [0]:
%pip install openpyxl --quiet

In [0]:
# ============================================================
# CELL 9 — Detect whether XLSX or CSV is present in the volume
#          Read with pandas → convert to Spark DataFrame
#
# openpyxl is pre-installed on Databricks Runtime 13+
# If you get ImportError: pip install openpyxl in a notebook cell
# ============================================================

xlsx_full_path = CATALOG_RAW_PATH + CATALOG_XLSX_FILE
csv_full_path  = CATALOG_RAW_PATH + CATALOG_CSV_FILE

# Check which file is available
# dbutils.fs.ls returns FileInfo objects; convert to list of file names
files_in_volume = [f.name for f in dbutils.fs.ls(CATALOG_RAW_PATH)]
print(f"Files found in volume: {files_in_volume}")

if CATALOG_XLSX_FILE in files_in_volume:
    # XLSX path — use pandas with openpyxl engine
    # Databricks Volumes are accessible at /Volumes/ path directly from pandas
    pandas_path = f"/Volumes/{CATALOG}/{BRONZE}/course_catalog_raw/{CATALOG_XLSX_FILE}"
    catalog_pandas_df = pd.read_excel(
        pandas_path,
        engine="openpyxl",
        dtype=str           # read all as string first — we cast types in Spark
    )
    source_file_used = CATALOG_XLSX_FILE
    print(f"✅ Read XLSX: {CATALOG_XLSX_FILE}")

elif CATALOG_CSV_FILE in files_in_volume:
    # CSV fallback
    pandas_path = f"/Volumes/{CATALOG}/{BRONZE}/course_catalog_raw/{CATALOG_CSV_FILE}"
    catalog_pandas_df = pd.read_csv(
        pandas_path,
        dtype=str           # read all as string first
    )
    source_file_used = CATALOG_CSV_FILE
    print(f"✅ Read CSV fallback: {CATALOG_CSV_FILE}")

else:
    raise FileNotFoundError(
        f"❌ Neither {CATALOG_XLSX_FILE} nor {CATALOG_CSV_FILE} found in {CATALOG_RAW_PATH}"
    )

# Clean column names — strip whitespace, lowercase
catalog_pandas_df.columns = [c.strip().lower().replace(" ", "_")
                              for c in catalog_pandas_df.columns]

print(f"   Rows  : {len(catalog_pandas_df):,}")
print(f"   Cols  : {list(catalog_pandas_df.columns)}")

In [0]:
# ============================================================
# CELL 10 — Convert pandas DataFrame to Spark
#           Cast columns to correct types
#           Add Bronze metadata columns
# ============================================================

# Convert to Spark DataFrame (all columns still string at this point)
catalog_spark_raw = spark.createDataFrame(catalog_pandas_df)

# Cast each column to the correct type
catalog_spark_df = (
    catalog_spark_raw
    .withColumn("total_modules",  F.col("total_modules").cast(IntegerType()))
    .withColumn("total_videos",   F.col("total_videos").cast(IntegerType()))
    .withColumn("total_quizzes",  F.col("total_quizzes").cast(IntegerType()))
    .withColumn("duration_hours", F.col("duration_hours").cast(DoubleType()))
    .withColumn("price_inr",      F.col("price_inr").cast(IntegerType()))
    .withColumn("is_active",      F.col("is_active").cast(BooleanType()).cast(IntegerType()))
    .withColumn("avg_rating",     F.col("avg_rating").cast(DoubleType()))
    .withColumn("launch_date",    F.to_date(F.col("launch_date")))
    # ── Bronze metadata ─────────────────────────────────────
    .withColumn("_source_file",      F.lit(source_file_used))
    .withColumn("_load_ts",          F.current_timestamp())
    .withColumn("_last_modified_ts", F.current_timestamp())
    .withColumn("_schema_version",   F.lit(SCHEMA_VERSION))
)

print(f"✅ Spark DataFrame ready")
print(f"   Rows   : {catalog_spark_df.count():,}")
print(f"   Schema :")
catalog_spark_df.printSchema()

In [0]:
# ============================================================
# CELL 11 — Write to course_catalog_bronze
#           Mode: OVERWRITE — full monthly refresh
#           Every run replaces the table with latest catalog file
# ============================================================

(
    catalog_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")   # allow schema changes between monthly files
    .saveAsTable(CATALOG_BRONZE_TABLE)
)

final_count = spark.table(CATALOG_BRONZE_TABLE).count()
print(f"✅ Written to: {CATALOG_BRONZE_TABLE}")
print(f"   Rows in table: {final_count:,}")

In [0]:
# ============================================================
# CELL 12 — Verify course_catalog_bronze
# ============================================================

cat_df = spark.table(CATALOG_BRONZE_TABLE)

print("── course_catalog_bronze ─────────────────────────")
print(f"Total rows   : {cat_df.count():,}")
print(f"Columns      : {len(cat_df.columns)}")
print()

print("── difficulty_level distribution ─────────────────")
cat_df.groupBy("difficulty_level").count().orderBy("difficulty_level").show()

print("── is_active distribution ────────────────────────")
cat_df.groupBy("is_active").count().show()

print("── avg_rating stats ──────────────────────────────")
cat_df.select(
    F.min("avg_rating").alias("min_rating"),
    F.max("avg_rating").alias("max_rating"),
    F.round(F.avg("avg_rating"), 2).alias("avg_rating")
).show()

print("── sample rows ───────────────────────────────────")
cat_df.select(
    "course_id", "course_title", "instructor_name",
    "difficulty_level", "is_active", "avg_rating"
).show(5, truncate=40)

## Part C — Data Quality Checks

DQ checks for this task:
1. No duplicate `course_id` in course_catalog_bronze
2. `is_active` must be 0 or 1 only (boolean check)
3. `avg_rating` must be between 0 and 5
4. `course_id` and `author_student_id` must not be NULL in discussion_bronze

All failures written to `audit.dq_audit`. Rows are NOT dropped from bronze.

In [0]:
# ============================================================
# CELL 14 — DQ Check 1: No duplicate course_ids in course_catalog
# A duplicate course_id would break all downstream joins
# This should never happen but must be caught and alerted
# ============================================================

cat_df = spark.table(CATALOG_BRONZE_TABLE)

duplicate_courses = (
    cat_df
    .groupBy("course_id")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)

dup_count = duplicate_courses.count()
print("── DQ Check 1: duplicate course_id ──────────────")
print(f"   Duplicate course_ids found: {dup_count}")

if dup_count > 0:
    print("   ⚠ Duplicates:")
    duplicate_courses.show(truncate=False)

    # Flag all affected rows in audit
    flagged = (
        cat_df
        .join(duplicate_courses.select("course_id"), on="course_id", how="inner")
        .withColumn("dq_check_name", F.lit("duplicate_course_id"))
        .withColumn("dq_table",      F.lit(CATALOG_BRONZE_TABLE))
        .withColumn("dq_severity",   F.lit("HIGH"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.concat_ws(" | ",
            F.lit("Duplicate course_id:"), F.col("course_id")
        ))
    )
    (
        flagged
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ Flagged rows written to {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED — no duplicate course_ids found")

In [0]:
# ============================================================
# CELL 15 — DQ Check 2: is_active must be 0 or 1 (boolean)
# Any other value (NULL, 2, -1, etc.) is invalid
# ============================================================

invalid_active = (
    spark.table(CATALOG_BRONZE_TABLE)
    .filter(
        F.col("is_active").isNull() |
        (~F.col("is_active").isin(0, 1))
    )
    .withColumn("dq_check_name", F.lit("invalid_is_active_value"))
    .withColumn("dq_table",      F.lit(CATALOG_BRONZE_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("is_active has invalid value:"),
        F.col("is_active").cast("string"),
        F.lit("for course_id:"), F.col("course_id")
    ))
)

invalid_active_count = invalid_active.count()
print("── DQ Check 2: is_active value validation ────────")
print(f"   Invalid is_active rows: {invalid_active_count}")

if invalid_active_count > 0:
    invalid_active.select("course_id", "is_active", "dq_message").show(5, truncate=False)
    (
        invalid_active
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_active_count} rows flagged → written to {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED — all is_active values are 0 or 1")

In [0]:
# ============================================================
# CELL 16 — DQ Check 3: avg_rating must be between 0.0 and 5.0
# NULL ratings are also flagged
# ============================================================

invalid_rating = (
    spark.table(CATALOG_BRONZE_TABLE)
    .filter(
        F.col("avg_rating").isNull() |
        (F.col("avg_rating") < 0) |
        (F.col("avg_rating") > 5)
    )
    .withColumn("dq_check_name", F.lit("invalid_avg_rating"))
    .withColumn("dq_table",      F.lit(CATALOG_BRONZE_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("avg_rating out of range [0,5]:"),
        F.col("avg_rating").cast("string"),
        F.lit("for course_id:"), F.col("course_id")
    ))
)

invalid_rating_count = invalid_rating.count()
print("── DQ Check 3: avg_rating range validation ───────")
print(f"   Invalid avg_rating rows: {invalid_rating_count}")

if invalid_rating_count > 0:
    invalid_rating.select("course_id", "avg_rating", "dq_message").show(5, truncate=False)
    (
        invalid_rating
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_rating_count} rows flagged → written to {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED — all avg_rating values are within [0, 5]")

In [0]:
# ============================================================
# CELL 17 — DQ Check 4: NULL course_id / author_student_id
#           in discussion_bronze
# These two fields are critical for all downstream Silver joins
# ============================================================

disc_df = spark.table(DISCUSSION_BRONZE_TABLE)
total   = disc_df.count()

for col_name in ["course_id", "author_student_id"]:
    null_rows  = disc_df.filter(F.col(col_name).isNull())
    null_count = null_rows.count()
    pct        = (null_count / total * 100) if total > 0 else 0
    status     = "✅" if null_count == 0 else "⚠"
    alert      = "  ← ALERT: exceeds 0.5% threshold!" if pct > 0.5 else ""

    print(f"── DQ Check 4: NULL {col_name} in discussion_bronze ──")
    print(f"   {status}  nulls={null_count:,}  ({pct:.3f}%){alert}")

    if null_count > 0:
        flagged = (
            null_rows
            .withColumn("dq_check_name", F.lit(f"null_{col_name}"))
            .withColumn("dq_table",      F.lit(DISCUSSION_BRONZE_TABLE))
            .withColumn("dq_severity",   F.lit("HIGH"))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.lit(f"{col_name} is NULL"))
        )
        (
            flagged
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "post_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {null_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    print()

In [0]:
%sql
-- ============================================================
-- CELL 18 — SQL summary of all 4 Bronze tables created so far
-- Run this after completing Tasks 1.1, 1.2, and 1.3
-- ============================================================

SELECT 'lms_events_bronze'    AS table_name, COUNT(*) AS total_rows FROM mini_project_grp6.bronze.lms_events_bronze
UNION ALL
SELECT 'video_watch_bronze',               COUNT(*) FROM mini_project_grp6.bronze.video_watch_bronze
UNION ALL
SELECT 'quiz_attempts_bronze',             COUNT(*) FROM mini_project_grp6.bronze.quiz_attempts_bronze
UNION ALL
SELECT 'enrollments_bronze',              COUNT(*) FROM mini_project_grp6.bronze.enrollments_bronze
UNION ALL
SELECT 'discussion_bronze',               COUNT(*) FROM mini_project_grp6.bronze.discussion_bronze
UNION ALL
SELECT 'course_catalog_bronze',           COUNT(*) FROM mini_project_grp6.bronze.course_catalog_bronze
ORDER BY table_name;

In [0]:
%sql
-- ============================================================
-- CELL 19 — Review all DQ flags written across Tasks 1.1–1.3
-- ============================================================

SELECT
    dq_table,
    dq_check_name,
    dq_severity,
    COUNT(*)          AS flagged_rows,
    MIN(dq_checked_at) AS first_flagged,
    MAX(dq_checked_at) AS last_flagged
FROM mini_project_grp6.audit.dq_audit
GROUP BY dq_table, dq_check_name, dq_severity
ORDER BY dq_severity, dq_table;

In [0]:
# ============================================================
# CELL 20 — Task 1.3 completion summary + full Bronze layer done
# ============================================================

disc_count = spark.table(DISCUSSION_BRONZE_TABLE).count()
cat_count  = spark.table(CATALOG_BRONZE_TABLE).count()

print("═" * 58)
print("  TASK 1.3 COMPLETE — Bronze Discussion + Catalog")
print("═" * 58)
print()
print(f"  ✅ {DISCUSSION_BRONZE_TABLE}")
print(f"      Rows    : {disc_count:,}")
print(f"      Method  : NDJSON read — typed fields + raw_json stored")
print(f"      Metadata: _source_file · _load_ts · _last_modified_ts")
print()
print(f"  ✅ {CATALOG_BRONZE_TABLE}")
print(f"      Rows    : {cat_count:,}")
print(f"      Method  : pandas.read_excel() → spark.createDataFrame()")
print(f"      Mode    : OVERWRITE (full monthly refresh)")
print(f"      Metadata: _source_file · _load_ts · _last_modified_ts")
print()
print("  DQ checks run:")
print("      Cell 14 — duplicate course_id in catalog")
print("      Cell 15 — is_active must be 0 or 1")
print("      Cell 16 — avg_rating between 0 and 5")
print("      Cell 17 — NULL course_id / author_student_id in discussion")
print()
print("  ─────────────────────────────────────────────────────")
print("  ✅ FULL BRONZE LAYER COMPLETE — all 6 tables loaded")
print()
print("  Bronze tables:")
print("      mini_project_grp6.bronze.lms_events_bronze")
print("      mini_project_grp6.bronze.video_watch_bronze")
print("      mini_project_grp6.bronze.quiz_attempts_bronze")
print("      mini_project_grp6.bronze.enrollments_bronze")
print("      mini_project_grp6.bronze.discussion_bronze")
print("      mini_project_grp6.bronze.course_catalog_bronze")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 2.1 → 04_silver_discussion_parsed")
print("         from_json() + post_depth + instructor_reply_rate")
print("═" * 58)